# African Wildlife Detector: YOLO end to end

Detect **buffalo, elephant, rhino, zebra** with a fine-tuned YOLO model.
Dataset: Ultralytics African Wildlife (1,504 images, pre-split into train/val/test). It downloads automatically.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Steps: baseline with the pretrained model → fine-tune → evaluate on the test split → study failures → export → demo.

In [ ]:
!pip -q install -U ultralytics gradio

In [ ]:
import torch, cv2, glob
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from ultralytics.utils import SETTINGS

print("GPU available:", torch.cuda.is_available())

## 1. Baseline: what does the pretrained model already know?

The pretrained model was trained on COCO. COCO has *elephant* and *zebra* but no *rhino*, and no buffalo.
Run it on a sample image and note what it gets wrong. That is the reason we fine-tune.

In [ ]:
base = YOLO("yolo11n.pt")          # if your ultralytics version has it, you can try "yolo26n.pt"
r = base.predict("https://ultralytics.com/assets/african-wildlife-sample.jpg", conf=0.25, verbose=False)[0]
plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.title("Pretrained (COCO) baseline"); plt.show()
print([r.names[int(c)] for c in r.boxes.cls])

## 2. Fine-tune on the wildlife dataset

About 50 epochs is enough for a first run. Training prints mAP each epoch.

In [ ]:
model = YOLO("yolo11n.pt")
model.train(data="african-wildlife.yaml", epochs=50, imgsz=640, batch=16, name="wildlife")

save_dir = Path(model.trainer.save_dir)
best = model.trainer.best
print("Run folder:", save_dir)
print("Best weights:", best)

## 3. Evaluate honestly on the held-out TEST split

The test images were never used for training or for picking the best epoch.

In [ ]:
metrics = YOLO(best).val(data="african-wildlife.yaml", split="test")
print(f"Test mAP50: {metrics.box.map50:.3f} | mAP50-95: {metrics.box.map:.3f}")
for name, m in zip(metrics.names.values(), metrics.box.maps):
    print(f"  {name:10s} mAP50-95: {m:.3f}")

In [ ]:
for f in ["results.png", "confusion_matrix.png"]:
    img = cv2.cvtColor(cv2.imread(str(save_dir / f)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 7)); plt.imshow(img); plt.axis("off"); plt.title(f); plt.show()

## 4. Look at predictions and find failures

Write down in your README where it fails: distant animals, partial views, herds, dim light, similar shapes.

In [ ]:
tuned = YOLO(best)
test_dir = Path(SETTINGS["datasets_dir"]) / "african-wildlife" / "images" / "test"
files = sorted(glob.glob(str(test_dir / "*.jpg")))[:12]
print(len(files), "images from", test_dir)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, f in zip(axes.ravel(), files):
    res = tuned.predict(f, conf=0.4, verbose=False)[0]
    ax.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Try your own photo
Upload an image (left sidebar → Files) and put its path below.

In [ ]:
my_image = "/content/my_photo.jpg"   # change this
if Path(my_image).exists():
    res = tuned.predict(my_image, conf=0.4, verbose=False)[0]
    plt.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
else:
    print("Upload a photo first, then re-run this cell.")

## 6. Export for deployment
ONNX runs without PyTorch. Also keep `best.pt` for the demo.

In [ ]:
onnx_path = tuned.export(format="onnx")
print("Exported:", onnx_path)

In [ ]:
# Save the weights to Google Drive so you don't lose them when the session ends
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/wildlife-detector
!cp "{best}" /content/drive/MyDrive/wildlife-detector/best.pt
!cp "{onnx_path}" /content/drive/MyDrive/wildlife-detector/

## 7. Quick demo (temporary link)
For a permanent demo, use `app.py` in a Hugging Face Space.

In [ ]:
import gradio as gr

def detect(image, conf):
    result = tuned.predict(image[:, :, ::-1], conf=conf, verbose=False)[0]   # RGB -> BGR in, BGR -> RGB out
    return result.plot()[:, :, ::-1]

gr.Interface(fn=detect,
             inputs=[gr.Image(type="numpy"), gr.Slider(0.1, 0.9, value=0.5, label="Confidence")],
             outputs=gr.Image(),
             title="African Wildlife Detector").launch(share=True)